# TP6 — Piloter un LLM depuis du code : classer des critiques de films (Énoncé)

Cours associé : [TP6_Cours.md](TP6_Cours.md) · Modèle local (facultatif) : [Installer_Ollama.md](../Installer_Ollama.md)

## Contexte

Un site de cinéma reçoit chaque jour des milliers de critiques écrites par ses membres. Il souhaite les classer automatiquement (positives ou négatives) et en extraire les points forts et les points faibles des films. Vous testez une solution à base de **grand modèle de langage** (LLM), appelé depuis Python, et vous la comparez à un modèle d'apprentissage automatique classique.

Le dossier `data/` contient des critiques du site Allociné :

| Fichier | Contenu |
|---|---|
| `allocine_train.csv` | 5 000 critiques étiquetées, pour entraîner le modèle classique |
| `allocine_test.csv` | 50 critiques courtes étiquetées, pour évaluer les deux approches |

Chaque fichier a deux colonnes : `critique` (le texte) et `sentiment` (`positif` ou `négatif`).

## Avant de commencer

**1. Les bibliothèques.** Dans l'environnement du cours :

```bash
uv pip install openai python-dotenv pydantic
```

ou, avec Conda :

```bash
conda install -c conda-forge openai python-dotenv pydantic
```

**2. Une clé d'API Mistral.** Sur [console.mistral.ai](https://console.mistral.ai), créez un compte, choisissez l'offre gratuite d'expérimentation (*Experiment*), puis créez une clé dans la rubrique *API Keys*. Copiez-la tout de suite : elle ne sera plus affichée.

**3. Le fichier `.env`.** Dans le dossier de ce TP, créez un fichier texte nommé `.env` (avec le point) contenant :

```bash
LLM_BASE_URL=https://api.mistral.ai/v1
LLM_API_KEY=collez-votre-clé-ici
LLM_MODEL=ministral-8b-latest
```

> ⚠️ Votre clé est **personnelle et secrète** : ne l'écrivez jamais dans le notebook, ne la partagez pas, ne publiez pas le fichier `.env` (ajoutez `.env` à votre `.gitignore`).

> 💡 Sans compte Mistral, vous pouvez faire tout le TP avec un modèle installé sur votre ordinateur : suivez [Installer_Ollama.md](../Installer_Ollama.md) et changez les trois lignes du `.env`. Les réponses seront moins bonnes et plus lentes.

## Déroulé

| Partie | Contenu |
|---|---|
| 1 | Premier appel : messages, rôles et absence de mémoire |
| 2 | Tokens et coût |
| 3 | La température |
| 4 | Ingénierie de prompt |
| 5 | Sorties structurées avec Pydantic |
| 6 | Évaluer le LLM et le comparer à un modèle classique |
| 7 | *Facultatif* : few-shot, autres modèles, extraction d'information |

Les cellules marquées `# TODO` sont à compléter. Les cellules ✍️ attendent une réponse rédigée.

Les réponses d'un LLM **varient** d'un appel à l'autre et d'une version du modèle à l'autre : ne vous étonnez pas d'obtenir des formulations différentes de celles de votre voisin.

In [ ]:
import os
import time
from typing import Literal

import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from openai import OpenAI, RateLimitError
from pydantic import BaseModel, Field

from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

load_dotenv()  # lit le fichier .env : la clé n'apparaît jamais dans le notebook
client = OpenAI(base_url=os.environ["LLM_BASE_URL"], api_key=os.environ["LLM_API_KEY"])
MODELE = os.environ["LLM_MODEL"]
print("Modèle utilisé :", MODELE)

La fonction `appeler` ci-dessous est **fournie**. Elle fait la même chose que `client.chat.completions.create` (ou `client.chat.completions.parse` quand on lui donne un schéma `response_format`, partie 5), mais si l'API répond que vous envoyez trop de requêtes (erreur 429, fréquente quand toute la classe travaille en même temps), elle attend quelques secondes et réessaie. Lisez-la, puis exécutez la cellule.

In [ ]:
def appeler(messages, modele=None, response_format=None, **parametres):
    """Envoie une liste de messages au modèle et renvoie la réponse complète de l'API.
    Avec un schéma Pydantic en response_format, la réponse est une sortie structurée (partie 5).
    Les paramètres supplémentaires (temperature, max_tokens…) sont transmis tels quels."""
    for essai in range(7):
        try:
            if response_format is None:
                return client.chat.completions.create(model=modele or MODELE, messages=messages, **parametres)
            return client.chat.completions.parse(model=modele or MODELE, messages=messages,
                                                 response_format=response_format, **parametres)
        except RateLimitError:
            attente = 2 ** essai
            print(f"Trop de requêtes : nouvel essai dans {attente} s")
            time.sleep(attente)
    raise RuntimeError("Limite de débit toujours dépassée : réessayez dans une minute.")

## Partie 1 — Premier appel : messages, rôles et absence de mémoire

**1.1** Appelez le modèle avec deux messages :
- un message `system` : « Tu es un assistant de l'IUT. Tu réponds en deux phrases au maximum. » ;
- un message `user` : « Qu'est-ce qu'un jeu de test en apprentissage automatique ? ».

Utilisez `temperature=0.2` et `max_tokens=150`. Affichez le texte de la réponse, la raison de l'arrêt (`finish_reason`) et la consommation de tokens (`usage`).

In [ ]:
# TODO : premier appel au modèle


**1.2** Relancez le même appel avec `max_tokens=10`. Que devient la réponse ? Que vaut `finish_reason` ?

In [ ]:
# TODO : même appel avec max_tokens=10


✍️ **Votre réponse :**

…

**1.3** Le modèle a-t-il une mémoire ? Faites **deux appels séparés** :
1. un message `user` : « Je m'appelle Léa et j'étudie l'informatique à Belfort. » ;
2. puis, dans un nouvel appel, un message `user` : « Comment je m'appelle et dans quelle ville j'étudie ? ».

Faites ensuite un **troisième appel** dont la liste `messages` contient tout l'échange : le premier message, la réponse du modèle (rôle `assistant`), puis la question.

In [ ]:
# TODO : deux appels séparés, puis un appel avec l'historique


✍️ **Votre réponse :**

…

**1.4** Écrivez une fonction `discuter(message)` qui ajoute le message de l'utilisateur à une liste globale `conversation`, appelle le modèle avec toute la liste, ajoute la réponse (rôle `assistant`) à la liste, et renvoie le texte de la réponse avec le nombre de tokens envoyés. Commencez la conversation par un message `system` de votre choix et faites trois tours de dialogue. Comment évolue le nombre de tokens envoyés ?

In [ ]:
# TODO : un mini-chatbot avec historique


✍️ **Votre réponse :**

…

## Partie 2 — Tokens et coût

La fonction `compter_tokens` ci-dessous est fournie : elle envoie un texte au modèle en limitant la réponse à un seul token, et lit le nombre de tokens envoyés dans `usage`. Ce nombre inclut quelques tokens de mise en forme du message, toujours les mêmes.

In [ ]:
def compter_tokens(texte):
    """Nombre de tokens envoyés pour un message contenant `texte` (mise en forme comprise)."""
    return compter_tokens_messages([{"role": "user", "content": texte}])


def compter_tokens_messages(messages):
    """Nombre de tokens envoyés pour une liste de messages complète."""
    return appeler(messages, max_tokens=1).usage.prompt_tokens

**2.1** Comptez les tokens des textes suivants et calculez, pour chacun, le nombre de caractères par token :
- « Le modèle de langage prédit le mot suivant. » ;
- sa traduction : « The language model predicts the next word. » ;
- « Le modele de langaje predi le mo suivan. » (la même phrase avec des fautes) ;
- « 3,14159265358979323846 ».

Que constatez-vous ?

In [ ]:
# TODO : tokens et caractères par token


✍️ **Votre réponse :**

…

**2.2** Chargez `data/allocine_test.csv` dans `test`. Prenez la critique la plus longue, comptez ses tokens et déduisez-en le nombre moyen de caractères par token pour du texte français ordinaire. Estimez ensuite le nombre de tokens d'une critique moyenne de `test`.

In [ ]:
# TODO : tokens d'une critique


**2.3** On veut classer les 20 000 critiques reçues chaque mois. Chaque appel envoie la critique plus une consigne d'environ 60 tokens, et reçoit une réponse d'environ 5 tokens. Calculez le coût mensuel avec les deux tarifs ci-dessous (indicatifs, novembre 2026, en dollars par million de tokens).

In [ ]:
TARIFS = {  # (prix d'un million de tokens envoyés, prix d'un million de tokens reçus), en $
    "ministral-8b-latest (petit modèle)": (0.15, 0.15),
    "mistral-large-latest (grand modèle)": (0.50, 1.50),
}

In [ ]:
# TODO : coût mensuel selon le modèle


✍️ **Votre réponse :**

…

## Partie 3 — La température

**3.1** Demandez cinq fois au modèle : « Propose un nom pour une application de covoiturage entre étudiants. Réponds uniquement par le nom. », d'abord avec `temperature=0`, puis avec `temperature=1.2` (`max_tokens=15`). Comparez.

In [ ]:
# TODO : même demande, deux températures


✍️ **Votre réponse :**

…

**3.2** Pour classer des critiques en positif ou négatif, quelle température choisir ? Et pour écrire automatiquement un résumé accrocheur de chaque film ? Justifiez.

✍️ **Votre réponse :**

…

## Partie 4 — Ingénierie de prompt

**4.1** Posez au modèle la question « Qu'est-ce qu'un token ? » (`temperature=0`, `max_tokens=80`), sans message `system`. Posez-la ensuite avec un message `system` qui lui donne un rôle adapté à ce cours. Comparez les deux réponses.

In [ ]:
# TODO : la même question, sans puis avec un rôle


✍️ **Votre réponse :**

…

**4.2** Voici une critique du jeu de test. Demandez au modèle de l'analyser avec un prompt **vague** : « Que penses-tu de cette critique ? » suivi de la critique. Écrivez ensuite un prompt **précis** qui :
- donne un rôle au modèle (message `system`) ;
- indique la tâche et les **seules réponses possibles** (`positif` ou `négatif`) ;
- place la critique entre des balises `<critique>` et `</critique>` ;
- demande de répondre **par un seul mot**.

Comparez les deux réponses : laquelle un programme pourrait-il utiliser ?

In [ ]:
critique = test.loc[test["critique"].str.startswith("Un nouveau film sur la mafia"), "critique"].iloc[0]
print(critique)

In [ ]:
# TODO : prompt vague, puis prompt précis


✍️ **Votre réponse :**

…

**4.3** Le prompt précis de 4.2 répond par du texte libre : rien n'empêche le modèle de répondre « Négatif. », « négative » ou « Je dirais négatif ». Testez votre prompt précis sur les cinq premières critiques de `test` et affichez les réponses brutes (avec `repr`, pour voir les espaces et la ponctuation). Les réponses sont-elles toutes directement comparables à la colonne `sentiment` ?

In [ ]:
# TODO : le prompt précis sur cinq critiques


✍️ **Votre réponse :**

…

## Partie 5 — Sorties structurées avec Pydantic

On veut maintenant extraire de chaque critique plusieurs informations, dans un format fixe. On les décrit par une classe Pydantic (cours, section 6).

**5.1** Complétez la classe `AnalyseCritique` avec quatre champs :
- `sentiment` : `"positif"` ou `"négatif"` (utilisez `Literal`) ;
- `note` : un entier, la note sur 5 que l'auteur aurait probablement donnée ;
- `points_forts` : une liste de chaînes (`list[str]`) ;
- `points_faibles` : une liste de chaînes.

Ajoutez à chaque champ une `description` (avec `Field`) qui explique au modèle ce qu'on attend.

In [ ]:
# TODO : le schéma de l'analyse d'une critique
class AnalyseCritique(BaseModel):

**5.2** Écrivez une fonction `analyser(texte)` qui appelle `appeler` avec `response_format=AnalyseCritique` et `temperature=0`, et renvoie l'objet `AnalyseCritique` obtenu (`reponse.choices[0].message.parsed`). Appliquez-la à la critique de 4.2, puis affichez le sentiment, la note et les points forts.

In [ ]:
# TODO : analyser une critique avec une sortie structurée


**5.3** Analysez les huit premières critiques de `test` et rassemblez les résultats dans un `DataFrame` (une ligne par critique ; `analyse.model_dump()` transforme l'objet en dictionnaire). Ajoutez la colonne du vrai sentiment. Les notes sont-elles cohérentes avec le sentiment ? Les points forts et faibles sont-ils fidèles aux critiques ?

In [ ]:
# TODO : un tableau d'analyses


✍️ **Votre réponse :**

…

## Partie 6 — Évaluer le LLM et le comparer à un modèle classique

**6.1** Pour classer, on n'a besoin que du sentiment. Définissez une classe `Sentiment` avec un seul champ `sentiment` (`Literal["positif", "négatif"]`), puis une fonction `classer(texte)` qui renvoie la chaîne `"positif"` ou `"négatif"` (même prompt qu'en 4.3, `temperature=0`). Classez les 50 critiques de `test` en mesurant le temps total (`time.perf_counter()`), et rangez les prédictions dans `pred_llm`.

In [ ]:
# TODO : classer les 50 critiques avec le LLM


**6.2** Évaluez ces prédictions comme au TP2 : exactitude, `classification_report` et matrice de confusion (`ConfusionMatrixDisplay.from_predictions`). Affichez ensuite les critiques mal classées. Les erreurs vous semblent-elles excusables ?

In [ ]:
# TODO : évaluation du LLM et erreurs


✍️ **Votre réponse :**

…

**6.3** Entraînez maintenant un modèle classique sur les 5 000 critiques de `data/allocine_train.csv` : un pipeline `TfidfVectorizer()` suivi d'une `LogisticRegression(max_iter=1000)`. Mesurez le temps d'entraînement et le temps de prédiction des 50 critiques de `test`, puis son exactitude et son rapport de classification.

In [ ]:
# TODO : TF-IDF + régression logistique


**6.4** Quels mots poussent le plus vers « positif » et vers « négatif » ? Affichez les 10 mots de plus grand coefficient et les 10 de plus petit (`logreg[0].get_feature_names_out()` et `logreg[-1].coef_[0]` ; la classe « positif » est la deuxième, `logreg[-1].classes_`).

In [ ]:
# TODO : les mots les plus influents


**6.5** Synthèse. Remplissez le tableau comparatif suivant (exactitude, temps par critique, coût, données nécessaires, confidentialité), puis répondez : pour le site de cinéma, qui reçoit 20 000 critiques par mois, que recommanderiez-vous ? Et pour une petite entreprise qui veut classer, dès demain, les avis sur ses produits en cinq catégories qu'elle vient d'inventer, sans aucune donnée étiquetée ?

✍️ **Votre réponse :**

…

## Partie 7 — *Facultatif* : few-shot, autres modèles, extraction d'information

**7.1** *Few-shot*. Écrivez une fonction `prompt_few_shot(texte)` qui reprend le prompt précis de 4.3 en ajoutant, avant la critique à classer, **quatre exemples résolus** tirés de `train` (deux positifs, deux négatifs), sous forme de paires de messages `user` / `assistant`. Classez à nouveau les 50 critiques. L'exactitude change-t-elle ? Combien de tokens coûte chaque appel en plus ?

In [ ]:
# TODO : few-shot avec quatre exemples


✍️ **Votre réponse :**

…

**7.2** *Autres modèles*. Les fonctions `classer` et `analyser` acceptent un paramètre `modele`. Classez les 50 critiques avec `ministral-3b-latest` (plus petit) et `ministral-14b-latest` (plus gros), en mesurant l'exactitude et le temps. Le plus gros modèle est-il toujours le meilleur choix ? (Avec l'offre gratuite, `ministral-14b-latest` n'accepte qu'une trentaine de requêtes par minute : la boucle prend environ deux minutes.)

*Avec Ollama, utilisez les noms de vos modèles locaux, par exemple `ministral-3:3b`.*

In [ ]:
# TODO : comparer plusieurs tailles de modèle


✍️ **Votre réponse :**

…

**7.3** *Extraction d'information*. Un service client reçoit des messages comme ceux ci-dessous. Écrivez une classe `Reclamation` avec les champs `commande` (numéro, **facultatif** : `int | None = None`), `produit`, `probleme` et `demande` (`Literal["remboursement", "échange", "réparation", "autre"]`), puis extrayez les informations des deux messages. Que se passe-t-il pour le numéro de commande du second message ? Retirez la valeur par défaut `None` et le type `None` du champ `commande` : que fait alors le modèle ?

In [ ]:
messages_clients = [
    "Bonjour, ma trottinette (commande 58213) est arrivée avec la batterie cassée. Je demande un remboursement. Camille M.",
    "Le casque audio que j'ai reçu hier grésille côté gauche. Pouvez-vous me l'échanger contre un neuf ? Merci, Sacha",
]

In [ ]:
# TODO : extraction d'information avec un champ facultatif


✍️ **Votre réponse :**

…

## Pour aller plus loin (bonus)

**Le *streaming*.** Avec `stream=True`, l'API envoie la réponse morceau par morceau, au fur et à mesure de sa génération, comme dans ChatGPT. Essayez :

```python
flux = client.chat.completions.create(model=MODELE, stream=True,
    messages=[{"role": "user", "content": "Raconte en dix lignes l'histoire d'un robot cinéphile."}])
for morceau in flux:
    print(morceau.choices[0].delta.content or "", end="", flush=True)
```

Dans une application web, c'est ce qui permet d'afficher la réponse sans attendre qu'elle soit complète : pensez-y pour votre projet.